# 04 · Generation and Grounding

The visible part, put in its place. Under a retrieval bottleneck a **bigger model barely
helps** — the answer is capped by what retrieval delivered (notebook 03). Where generation
*does* matter is **faithfully using the provided context**, which we measure as
**groundedness** — distinct from correctness.


In [1]:
import os, sys, json
from pathlib import Path

# Run from the repo root so outputs/ and data/ resolve no matter where the notebook starts.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import support
from src.config import load_settings
from src.datasets.corpus import build_corpus
from src.pipeline.embed import build_embedder

settings = load_settings()
corpus = build_corpus()
embedder = build_embedder(settings.embed)
generator, mode = support.select_generator(corpus, settings)
print("generator mode:", mode)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

generator mode: ollama:qwen2.5:0.5b


### Model swap — hold chunking/retrieval fixed, change only the generator

In [2]:
from src.evaluation.ablations import model_swap
from src.pipeline.generate import OllamaGenerator

if mode.startswith("ollama"):
    generators = {
        "qwen2.5:0.5b": OllamaGenerator(settings.generate.model_copy(update={"model_name": "qwen2.5:0.5b"})),
        "qwen2.5:1.5b": OllamaGenerator(settings.generate.model_copy(update={"model_name": "qwen2.5:1.5b"})),
    }
else:
    # Offline: two stand-ins of different "strength". Both are capped by retrieval recall,
    # which is exactly the point — the model is not the bottleneck.
    generators = {"tiny (stand-in)": generator, "larger (stand-in)": generator}

swap_df = model_swap(corpus, embedder, generators, settings)
swap_df


,model,retrieval_recall,answer_accuracy
0,qwen2.5:0.5b,1.0,1.000000
1,qwen2.5:1.5b,1.0,0.833333


### Groundedness vs correctness — they are not the same thing

In [3]:
from src.datasets.corpus import chunk_corpus
from src.pipeline.chunk import make_chunk_fn
from src.pipeline.retrieve import build_retriever
from src.pipeline.rag import RagPipeline
from src.evaluation.answer_metrics import answer_correct
from src.grounding.groundedness import groundedness_score

cf = make_chunk_fn(settings.chunk.strategy, size=settings.chunk.size,
                   overlap=settings.chunk.overlap, encoder=embedder)
chunks, chunk_doc_ids = chunk_corpus(corpus, cf)
retriever = build_retriever(settings.retrieve.method, embedder, settings.retrieve.hybrid_alpha)
rag = RagPipeline(retriever=retriever, generator=generator, settings=settings, abstain=False)
rag.index(chunks)

rows = []
for q in corpus.questions_by_kind("answerable"):
    r = rag.answer(q.text)
    rows.append({"question": q.text, "answer": r.answer, "gold": q.answer,
                 "correct": bool(answer_correct(r.answer, q.answer)),
                 "groundedness": round(groundedness_score(r.answer, r.contexts), 2)})
grounding_df = pd.DataFrame(rows)
grounding_df


,question,answer,gold,correct,groundedness
0,What is the capital city of Volara?,The capital city of Volara is Thornmouth.,Thornmouth,True,1.0
1,How tall is the Graywing windmill?,The Graywing windmill stands 62 metres high.,62 metres,True,1.0
2,What currency does the Zephyr Archipelago use?,The Zephyr Archipelago uses the zeph currency.,the zeph,True,1.0
3,When did Mount Sear last erupt?,1911,1911,True,1.0
4,How many members does the Archipelago Assembly...,The Archipelago Assembly has 60 members.,60 members,True,1.0
5,Where does the silver tern nest?,The silver tern nests exclusively on the cliff...,the cliffs of Pelagos,True,1.0


### Figures → PNG, tables → JSON

In [4]:
from src.visualisation import bar_chart

fig = bar_chart(swap_df["model"], swap_df["answer_accuracy"],
                title="Answer accuracy by model (retrieval fixed)",
                ylabel="answer accuracy", ylim=(0, 1.05))
p = support.save_fig(fig, "04_model_swap")
support.save_json({"mode": mode,
                   "model_swap": swap_df.to_dict(orient="records"),
                   "grounding": grounding_df.to_dict(orient="records")}, "04_generation")
print("saved", p)
fig


saved outputs/figures/04_model_swap.png


<Figure size 700x400 with 1 Axes>